# NB_SLV_PN_POI

RECOMPUTE `lh_vv_silver.dbo.slv_pn_poi`. Gold chỉ copy `poi_id`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:

def build(ctx: NodeContext) -> DataFrame:
    # ID-004 — MQ-006, MQ-063, AGENTS.md §9. poi_id partner tính ở silver.
    # DS-013 — MQ-063. business_service_id copy id.
    # name — decision.md 2026-10-13, e2. locale vi. Không có dòng vi thì NULL.
    # short_address — MQ-025, e1. address.full_address
    # ward — decision.md 2026-10-13, e1. address.ward
    # district — decision.md 2026-10-13. address.district, kể cả chuỗi rỗng
    # province_name — decision.md 2026-10-13, e1. address.city
    vi = (ctx.input("i18n").where(F.col("locale") == "vi")
          .select(F.col("business_service_id").alias("vi_id"), F.col("name").alias("vi_name")))
    bs = ctx.input("bs").join(vi, F.col("id") == F.col("vi_id"), "left")
    return bs.select(
        uuid_md5(F.lit("partner_portal"), F.col("id")).alias("poi_id"),
        F.col("id").alias("business_service_id"),
        F.col("vi_name").alias("name"),
        F.get_json_object("address", "$.full_address").alias("short_address"),
        F.get_json_object("address", "$.ward").alias("ward"),
        F.get_json_object("address", "$.district").alias("district"),
        F.get_json_object("address", "$.city").alias("province_name"),
    )


SPEC = NodeSpec(
    trg_tbl="slv_pn_poi",
    inputs={"bs": "slv_pn_business_services", "i18n": "slv_pn_business_service_i18ns"},
    pk=("poi_id",),
    build=build,
    required_inputs=("bs",),
    unique_checks=(("business_service_id_unique", ("business_service_id",), None),),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
